In [65]:
import pickle
import pandas as pd
import numpy as np
from scipy.integrate import simpson
from scipy.stats import gaussian_kde
from scipy.stats import norm

In [71]:
with open('/home/yaning/Documents/Discounting/paper/results/gamble_samples.pkl', 'rb') as f:
    pos_dict = pickle.load(f)

In [72]:
samples = pos_dict["locs"]

In [73]:
average = np.mean(samples, axis=1)

In [74]:
average.shape

(1000, 3)

In [75]:
gamble = average

In [7]:
def get_the_BFkl(cafe, gamble):
    # get the density functions of both
    kde_cafe = gaussian_kde(cafe, bw_method='scott')
    kde_gamble = gaussian_kde(gamble, bw_method='scott')

    # sample from both density functions
    cafe_samples = kde_cafe.resample(size=1000)
    gamble_samples = kde_gamble.resample(size=1000)

    # difference between the samples (central something theory)
    diff = cafe_samples - gamble_samples
    diff = diff.reshape(1000)

    # get the density function of the difference
    kde_diff = gaussian_kde(diff, bw_method='scott')

    x = np.linspace(diff.min(), diff.max(), 1000)
    if x.min() > 0:
        mode = 'all pos'
        return mode
    elif x.max() < 0:
        mode = 'all neg'
        return mode

    x_pos = x[x > 0]
    x_neg = x[x < 0]

    y_pos = kde_diff(x_pos)
    y_neg = kde_diff(x_neg)

    inte_pos = simpson(y_pos, x = x_pos)
    inte_neg = simpson(y_neg, x = x_neg)

    if inte_pos > inte_neg:
        BFkl =  inte_pos / inte_neg if inte_neg != 0 else np.inf
        mode = 'pos bigger'
    else:
        BFkl =  inte_neg / inte_pos if inte_pos != 0 else np.inf
        mode = 'neg bigger'

    return {BFkl, mode}

In [81]:
test_cafe = np.exp(cafe[:,0])/(1+np.exp(cafe[:,0])*np.exp(-5))

In [86]:
gamble[:,2].mean()

np.float32(2.344674)

In [87]:
cafe[:,2].mean()

np.float32(2.4199164)

In [82]:
test_gamble = np.exp(gamble[:,0])/(1+np.exp(gamble[:,0])*np.exp(-5))

In [83]:
get_the_BFkl(test_cafe, test_gamble)

'all neg'

In [181]:
df = pd.DataFrame(average, columns=['sigma_rate', 'exp_a', 'exp_b', 'beta'])

In [182]:
df.to_csv("/home/yaning/Documents/Discounting/paper/results/future_sample.csv", index=False)